# Readabiilty Evaluation Framework Comparison Part 2

Compare Readability Scores as frameworks in and of themselves, comparing to the chosen ground truth CAREC score. Find the scores that are most highly correlated with the other scores, and see which scores are most similar to CAREC.

https://www.commonlit.org/blog/introducing-the-clear-corpus-an-open-dataset-to-advance-research-28ff8cfea84a/

In [6]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [36]:
clear = pd.read_csv('../data/CLEAR_readability_original.csv')
clear.head().T

,0,1,2,3,4
ID,400,401,402,403,404
Last Changed,NaN,NaN,NaN,NaN,NaN
Author,Carolyn Wells,Carolyn Wells,Carolyn Wells,CHARLES KINGSLEY,Charles Kingsley
Title,Patty's Suitors,Two Little Women on a Holiday,Patty Blossom,THE WATER-BABIES\nA Fairy Tale for a Land-Baby,HOW THE ARGONAUTS WERE DRIVEN INTO THE UNKNOWN...
Anthology,NaN,NaN,NaN,NaN,The Heroes\n or Greek Fairy Tales for my...
URL,http://www.gutenberg.org/cache/epub/5631/pg563...,http://www.gutenberg.org/cache/epub/5893/pg589...,http://www.gutenberg.org/cache/epub/20945/pg20...,http://www.gutenberg.org/files/25564/25564-h/2...,http://www.gutenberg.org/files/677/677-h/677-h...
Source,gutenberg,gutenberg,gutenberg,gutenberg,gutenberg
Pub Year,1914.0,1917.0,1917.0,1863.0,1889.0
Category,Lit,Lit,Lit,Lit,Lit
Location,mid,mid,mid,mid,mid


Filter scores of note. Leave off Grade Level scores (which are themselves translations of other scores), CAREC-M (derived from CAREC) as well as BT Easiness (Bradley-Terry coefficient derived from human annotated reading scores - not a readability evaluation framework).

In [37]:
clear = clear[['CAREC', 'Flesch-Reading-Ease', 'Automated Readability Index', 'SMOG Readability', 'New Dale-Chall Readability Formula', 'CARES', 'CML2RI']]
clear.head()

,CAREC,Flesch-Reading-Ease,Automated Readability Index,SMOG Readability,New Dale-Chall Readability Formula,CARES,CML2RI
0,0.12102,81.70,7.37,8.0,6.55,0.457534,12.097815
1,0.04921,80.26,4.16,7.0,6.25,0.462510,22.550179
2,0.10172,79.04,5.81,9.0,7.31,0.369259,18.125279
3,0.07491,44.77,24.87,12.0,8.56,0.390759,10.959460
4,0.06356,68.07,15.47,8.0,7.00,0.389226,3.195960


Scale each score to be between 0 and 100. Though some scores are whole numbers and some are not, and this will allow for a more direct comparison.

In [38]:
scaler = MinMaxScaler(feature_range=(0, 100))
clear[clear.columns] = scaler.fit_transform(clear[clear.columns])
clear.head()

,CAREC,Flesch-Reading-Ease,Automated Readability Index,SMOG Readability,New Dale-Chall Readability Formula,CARES,CML2RI
0,37.672499,77.394770,19.129481,44.444444,45.075485,49.618045,31.938190
1,28.323699,76.387918,13.258961,38.888889,42.918763,50.361396,52.196419
2,35.159871,75.534890,16.276518,50.000000,50.539180,36.431456,43.620307
3,31.669531,51.573207,51.133870,66.666667,59.525521,39.643165,29.731889
4,30.191897,67.864634,33.942941,44.444444,48.310568,39.414248,14.685075


### Check correlation with CAREC

In [39]:
corr_matrix = clear.corr()
carec_corr = corr_matrix['CAREC'].drop('CAREC')

In [40]:
carec_corr_sorted = carec_corr.abs().sort_values(ascending=False)
print(carec_corr_sorted)

New Dale-Chall Readability Formula    0.748200
Flesch-Reading-Ease                   0.722205
SMOG Readability                      0.714882
CARES                                 0.688305
CML2RI                                0.576794
Automated Readability Index           0.501597
Name: CAREC, dtype: float64


### Check Error with CAREC

In [41]:
clear_error = pd.DataFrame()

for col in clear.columns:
    clear_error[col] = clear[col] - clear['CAREC']

clear_error.drop('CAREC', axis=1, inplace=True)
clear_error.head()


,Flesch-Reading-Ease,Automated Readability Index,SMOG Readability,New Dale-Chall Readability Formula,CARES,CML2RI
0,39.722271,-18.543018,6.771945,7.402986,11.945546,-5.734309
1,48.064218,-15.064738,10.565189,14.595064,22.037697,23.872719
2,40.375019,-18.883353,14.840129,15.379310,1.271585,8.460437
3,19.903676,19.464339,34.997136,27.855990,7.973634,-1.937642
4,37.672737,3.751044,14.252547,18.118671,9.222351,-15.506822


In [42]:
mae = clear_error.abs().mean().sort_values()
mae

CARES                                 11.211315
New Dale-Chall Readability Formula    11.697089
SMOG Readability                      15.226672
Automated Readability Index           19.882765
CML2RI                                21.176072
Flesch-Reading-Ease                   27.959978
dtype: float64

### What Readability Scores are most correlated with the other scores?

In [45]:
corr_matrix = clear.corr()

avg_corr = corr_matrix.abs().mean().sort_values(ascending=False)
avg_corr

Flesch-Reading-Ease                   0.806728
SMOG Readability                      0.783770
New Dale-Chall Readability Formula    0.760142
CAREC                                 0.707426
Automated Readability Index           0.684222
CML2RI                                0.667918
CARES                                 0.614916
dtype: float64